In [10]:
import jax
import jax.numpy as jnp
import numpy as np
from tca_openflux import TCA_OpenFLux, objective_wssr, objective_clr_wssr, flux_vector, calc_ci
from emu_example import closure, clr_func, clr_inv_func

import optimistix
import matplotlib.pyplot as plt

from scipy import stats


In [23]:
def fva(parameters: dict[str, float]) -> dict:
    v_free = {"v8": 0.215, "v1": 1, "v19": 0.05, "v20": 0.03, "v16": 0.07, "v17": 0.23, "v18": 0.12, "v2": 0.201, "v9": 0.296}
    # The free fluxes are split into fixed fluxes and parameters, where parameters are the one being estimated. 
    fixed_keys = ["v1", "v16", "v17", "v18", "v19", "v20"]
    fixed = {k: v_free[k] for k in fixed_keys}

    # We calculate the true data (MIDs)
    true_data = TCA_OpenFLux(parameters=parameters, fixed=fixed)

    key = jax.random.PRNGKey(1234)
    sims = 1000

    total_mol = 1e5
    total_isos = jax.tree.map(lambda x: x * total_mol, true_data)
    cv = 0.2
    norm_sigma_cv = jax.tree.map(lambda x: x * cv, total_isos)

    key, key_sim = jax.random.split(key)
    # We assume the absolute abandance to be independent normal distributions. 
    norm_sim = jax.tree.map(
        lambda mean, sigma: mean + sigma * jax.random.normal(key_sim, (sims, mean.shape[0])),
        total_isos,
        norm_sigma_cv,
    )

    # Simulated absolute abandance are closed to 1 to make relative abundance
    iso_proportions = jax.tree.map(
        lambda mat: closure(mat),
        norm_sim,
    )

    # We calculate the mean of the relative abundances. 
    mean_sim = jax.tree.map(
        lambda x: jnp.mean(x, axis=0),  # mean over isotopolouges
        iso_proportions
    )
    # and the standard deviation
    std_sim = jax.tree.map(
        lambda x: jnp.std(x, axis=0),  # standard deviation over isotopolouges
        iso_proportions
    )

    # and compute a covariance matrix
    D_sim = jax.tree.map(
        lambda x: jnp.diag(x), 
        std_sim
    )

    # We specify the solver and initial parameters
    solver = optimistix.BFGS(
        rtol=1e-12, atol=1e-12, verbose=frozenset({"step"})
    )

    init_params = {
        "v8": 1.0,
        "v2": 1,
        "v9": 1,
    }

    # 
    result_optimistix = optimistix.minimise(objective_wssr, y0=init_params, args=(mean_sim, D_sim, fixed), solver=solver, max_steps=10000)

    # CLR
    iso_clr = jax.tree.map(
        lambda x: clr_func(x),
        iso_proportions
    )

    mean_clr = jax.tree.map(
        lambda x: jnp.mean(x, axis=0),  # mean over bins
        iso_clr
    )

    std_clr = jax.tree.map(
        lambda x: jnp.std(x, axis=0),  # mean over isotopolouges
        iso_clr
    )

    D_clr = jax.tree.map(
        lambda x: jnp.diag(x), 
        std_clr
    )

    result_optimistix_clr  = optimistix.minimise(objective_clr_wssr, y0=init_params, args=(mean_clr, D_clr, fixed), solver=solver, max_steps=10000)
    return {"Euclidean": result_optimistix.value, "Aitchison": result_optimistix_clr.value}

In [12]:
parameters = {"v8": [0.200, 0.210, 0.220], "v2": [0.190, 0.20, 0.210], "v9": [0.190, 0.20, 0.210]}
v_true_dict = []
v_est_dict = []
v_clr_dict = []
for param_set in (
    dict(zip(parameters, values))
    for values in zip(*parameters.values())
):
    v_true_dict.append(param_set) 
    v_est_dict.append(fva(param_set)["Euclidean"])
    v_clr_dict.append(fva(param_set)["Aitchison"])